# Synthetic walk-forward signal

This notebook is a deterministic, offline introduction to MFDRO. It constructs daily simple returns, validates the complete formation schedule, estimates a projected-quantile path, inspects its audit trail, and verifies a portable save/load round trip.

The data are deliberately synthetic: the objective is to understand the package contract, not to make an empirical claim.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import mfdro
from mfdro import MultiFrequencySignal, SignalConfig, SignalPath

plt.style.use("seaborn-v0_8-whitegrid")
INK = "#183b4e"
ACCENT = "#008c82"
print(f"MFDRO {mfdro.__version__}")

## 1. Construct a labelled daily return panel

MFDRO expects rows of unique increasing observation dates, columns of unique asset identifiers, and numeric **simple returns**. The fixed random seed makes this example exactly repeatable.

In [ ]:
rng = np.random.default_rng(20250301)
dates = pd.bdate_range("2018-01-01", "2022-12-30")
common_factor = rng.normal(0.0002, 0.008, size=(len(dates), 1))
idiosyncratic = rng.normal(0.0, 0.006, size=(len(dates), 6))
returns = pd.DataFrame(
    common_factor + idiosyncratic,
    index=dates,
    columns=[f"asset_{index:02d}" for index in range(6)],
)

returns.describe().T

## 2. Choose and preserve a configuration

The projected preset is intended for exploration and pipeline checks. Its name makes clear that the center is constructed direction by direction rather than stored as one multivariate free-support barycenter. `SignalConfig.reference()` exposes the declared free-support reference settings; notebook 03 studies that geometry directly.

In [ ]:
config = SignalConfig.projected(
    n_projections=64,
    n_quantiles=64,
    random_state=20250301,
)
reference_config = SignalConfig.reference()
engine = MultiFrequencySignal(config)

print(config.to_json())
pd.DataFrame(
    {
        "center": [config.barycenter, reference_config.barycenter],
        "distance": [config.distance, reference_config.distance],
        "projections": [config.n_projections, reference_config.n_projections],
        "digest": [config.digest[:12], reference_config.digest[:12]],
    },
    index=["exploration preset", "reference preset"],
)

## 3. Validate before computing geometry

Preflight uses the same rolling-window and frequency construction logic as estimation. Initial months are normally unavailable while the 36-month lookback warms up.

In [ ]:
diagnostics = engine.validate_path_inputs(
    returns,
    lookback_months=36,
    seed_namespace="synthetic_notebook",
)

assert diagnostics.is_usable
diagnostics.summary(), diagnostics.formations.head()

## 4. Estimate with an optional progress callback

The callback is deliberately dependency-free. A notebook UI or `tqdm` adapter can consume the same immutable updates.

In [ ]:
progress = []
path = engine.estimate_path(
    returns,
    lookback_months=36,
    on_insufficient="skip",
    seed_namespace="synthetic_notebook",
    progress_callback=progress.append,
)

assert len(progress) == diagnostics.n_formations
assert len(path.estimates) == diagnostics.n_ready
path.summary()

In [ ]:
path.estimates[["date", "rho", "sqrt_rho", "n_assets"]].tail()

In [ ]:
path.audit[["date", "n_daily", "n_weekly", "n_monthly", "no_future_observations"]].tail()

In [ ]:
figure, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
axes[0].plot(path.sqrt_rho.index, path.sqrt_rho, color=INK, linewidth=1.8)
axes[0].set(title="Synthetic multi-frequency dispersion", ylabel=r"$\sqrt{\rho}$")
for column, color in zip(
    ["n_daily", "n_weekly", "n_monthly"], [INK, ACCENT, "#d17a22"], strict=True
):
    axes[1].plot(
        path.audit["date"], path.audit[column], color=color, label=column.removeprefix("n_")
    )
axes[1].set(xlabel="formation date", ylabel="observations per measure")
axes[1].legend(frameon=False, ncol=3)
figure.tight_layout()

## 5. Verify the portable artifact

The saved directory contains the three result tables, exact configuration, manifest, and checksums. It does not use pickle.

In [ ]:
with TemporaryDirectory() as temporary_directory:
    destination = path.save(Path(temporary_directory) / "signal_path")
    restored = SignalPath.load(destination)
    pd.testing.assert_frame_equal(restored.estimates, path.estimates)
    assert restored.config.digest == config.digest

print("Portable round trip verified.")

## Interpretation boundary

A larger `rho` means the configured, scaled empirical frequency measures disagree more strongly around their configured center. It does not predict direction and is not automatically a portfolio ambiguity radius. A downstream calibration and backtest must define that mapping point in time.